# Plots iniciais do artigo: losses, métricas e comparação de modelos

Este notebook **não retreina modelos** e **não recalcula modelos**. Ele apenas importa os resultados já salvos em `experimentos_pinn/`, seleciona o subconjunto de arquiteturas que você quer mostrar e gera plots protótipos com visual próximo ao Mathematica/Wolfram.

Estrutura esperada:

```text
experimentos_pinn/
    sumario_classico.csv
    sumario_quantico.csv
    sumario_cquantico.csv
    sumario_hibrido.csv
    runs/<MODEL_TYPE>/<RUN_ID>/losses/loss_history_full.json
    runs/<MODEL_TYPE>/<RUN_ID>/metadata/results.json
```

A célula **Seleção dos experimentos já rodados** é a parte principal: ali você escolhe quais `hidden`, `blocks`, `n_qubits`, `n_layers` e limites de parâmetros entram nos plots.

Primeiro fazemos: loss total, decomposição da loss, erro por família, erro versus parâmetros, Pareto, tempo versus erro e heatmaps de arquitetura. Depois entram preço, mercado e Greeks.


In [ ]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, MaxNLocator, LogLocator
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# ESTILO GRÁFICO TIPO MATHEMATICA/WOLFRAM
# ============================================================

WOLFRAM_COLORS = [
    "#5E81B5",  # blue
    "#E19C24",  # orange
    "#8FB131",  # green
    "#C44E52",  # red
    "#80699B",  # purple
    "#51A7C9",  # cyan
    "#8C6D31",  # brown
    "#7F7F7F",  # gray
]

WOLFRAM_TEMPERATURE = LinearSegmentedColormap.from_list(
    "WolframTemperatureLike",
    ["#313695", "#4575B4", "#74ADD1", "#ABD9E9", "#FFFFBF", "#FEE090", "#F46D43", "#D73027", "#A50026"],
)

def set_mathematica_style(font_size=12):
    mpl.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.05,
        "figure.dpi": 130,
        "savefig.dpi": 300,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif", "STIXGeneral"],
        "mathtext.fontset": "stix",
        "font.size": font_size,
        "axes.titlesize": font_size + 1,
        "axes.labelsize": font_size,
        "xtick.labelsize": font_size - 1,
        "ytick.labelsize": font_size - 1,
        "legend.fontsize": font_size - 1,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.9,
        "axes.grid": True,
        "grid.color": "0.84",
        "grid.linewidth": 0.55,
        "grid.alpha": 1.0,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.minor.size": 2,
        "ytick.minor.size": 2,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "lines.linewidth": 1.7,
        "lines.markersize": 4.2,
        "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=WOLFRAM_COLORS),
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

def finish_axes(ax, xlabel=None, ylabel=None, title=None, logy=False):
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, pad=8)
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.9)
        ax.spines[side].set_color("black")
    ax.tick_params(which="both", top=True, right=True)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    if not logy:
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    if not logy:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

def savefig(fig, name):
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    path_pdf = FIG_DIR / f"{name}.pdf"
    path_png = FIG_DIR / f"{name}.png"
    fig.savefig(path_pdf)
    fig.savefig(path_png)
    print(f"salvo: {path_pdf}")
    return path_pdf

set_mathematica_style(font_size=12)


In [ ]:

# ============================================================
# CONFIGURAÇÃO
# ============================================================

# Rode este notebook na raiz do projeto.
# Se necessário, troque PROJECT_ROOT manualmente.
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "experimentos_pinn"
RUNS_DIR = RESULTS_DIR / "runs"
FIG_DIR = PROJECT_ROOT / "figures_loss_mathematica_style"
TABLE_DIR = PROJECT_ROOT / "tables_loss_mathematica_style"

SUMMARY_FILES = {
    "classico": RESULTS_DIR / "sumario_classico.csv",
    "quantico": RESULTS_DIR / "sumario_quantico.csv",
    "cquantico": RESULTS_DIR / "sumario_cquantico.csv",
    "hibrido": RESULTS_DIR / "sumario_hibrido.csv",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR existe?", RESULTS_DIR.exists(), RESULTS_DIR)
print("RUNS_DIR existe?", RUNS_DIR.exists(), RUNS_DIR)
print("\nSumários encontrados:")
for name, path in SUMMARY_FILES.items():
    print(f"  {name:10s} | {path.exists()} | {path}")

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"Não encontrei {RESULTS_DIR}. Rode o notebook na raiz do projeto ou ajuste PROJECT_ROOT."
    )


In [ ]:

# ============================================================
# LOADERS FIÉIS À ESTRUTURA DO PROJETO
# ============================================================

TEXT_COLS = {
    "run_id", "model_type", "model_label", "activation", "entangler", "model_class",
    "source_summary", "source_family", "model_path", "loss_history_path", "config_path",
    "json_path", "run_dir", "device", "run_id_prefix", "loss_path", "results_path"
}

METRIC_CANDIDATES = [
    "mse_teste_desnormalizado",
    "mse_teste_normalizado",
    "final_loss_total",
    "mean_last_100_loss_total",
    "final_loss",
    "mean_last_100_loss",
]

def maybe_numeric_series(s: pd.Series) -> pd.Series:
    # Converte coluna para numérico somente quando a conversão faz sentido.
    if s.name in TEXT_COLS:
        return s
    if pd.api.types.is_numeric_dtype(s):
        return s
    converted = pd.to_numeric(s, errors="coerce")
    original_non_null = int(s.notna().sum())
    converted_non_null = int(converted.notna().sum())
    if original_non_null > 0 and converted_non_null / original_non_null >= 0.80:
        return converted
    return s

def normalize_family(row) -> str:
    mt = str(row.get("model_type", "")).upper()
    src = str(row.get("source_summary", "")).lower()
    rid = str(row.get("run_id", "")).lower()
    if mt in {"MLP", "RESNET"} or "classico" in src or rid.startswith("mlp"):
        return "Clássico"
    if mt in {"QNN", "QPINN"} or "quantico" in src or rid.startswith("qnn"):
        return "QNN"
    if mt in {"CQNN", "CQNN_NONLINEAR"} or "cquantico" in src or rid.startswith("cqnn"):
        return "CQNN"
    if mt in {"HQNN"} or "hibrido" in src or rid.startswith("hqnn"):
        return "Híbrido"
    return mt if mt else "desconhecido"

def model_signature(row) -> str:
    fam = row.get("model_family", normalize_family(row))
    mt = str(row.get("model_type", ""))
    parts = [fam]
    if pd.notna(row.get("hidden", np.nan)):
        parts.append(f"h={int(row['hidden'])}")
    if pd.notna(row.get("blocks", np.nan)):
        parts.append(f"b={int(row['blocks'])}")
    if pd.notna(row.get("n_qubits", np.nan)):
        parts.append(f"q={int(row['n_qubits'])}")
    if pd.notna(row.get("n_layers", np.nan)):
        parts.append(f"L={int(row['n_layers'])}")
    if pd.notna(row.get("k", np.nan)):
        parts.append(f"k={int(row['k'])}")
    if pd.notna(row.get("n_vertex", np.nan)):
        parts.append(f"v={int(row['n_vertex'])}")
    if len(parts) == 1:
        parts.append(mt)
    return " | ".join(parts)

def choose_metric_column(df: pd.DataFrame) -> str:
    for c in METRIC_CANDIDATES:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").notna().any():
            return c
    numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    raise ValueError(
        "Não encontrei coluna de métrica. Colunas numéricas disponíveis: " + ", ".join(numeric)
    )

def read_summary_csv(path: Path, source_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["source_summary"] = source_name
    for col in df.columns:
        df[col] = maybe_numeric_series(df[col])
    return df

def crawl_results_json(results_dir: Path) -> pd.DataFrame:
    rows = []
    for p in sorted((results_dir / "runs").glob("*/*/metadata/results.json")):
        try:
            with open(p, "r", encoding="utf-8") as f:
                row = json.load(f)
        except Exception:
            continue
        row["results_path"] = str(p)
        row["run_dir"] = str(p.parents[1])
        row.setdefault("run_id", p.parents[1].name)
        row.setdefault("model_type", p.parents[2].name)
        row["source_summary"] = "metadata/results.json"
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    for col in df.columns:
        df[col] = maybe_numeric_series(df[col])
    return df

def load_all_summaries(results_dir: Path) -> pd.DataFrame:
    frames = []
    for source_name, path in SUMMARY_FILES.items():
        if path.exists():
            frames.append(read_summary_csv(path, source_name))
    # Se algum sumário não existe, ainda assim busca metadata/results.json.
    meta = crawl_results_json(results_dir)
    if not meta.empty:
        frames.append(meta)
    if not frames:
        raise FileNotFoundError(
            "Não encontrei sumários nem metadata/results.json dentro de experimentos_pinn."
        )
    df = pd.concat(frames, ignore_index=True, sort=False)

    # Remove duplicatas mantendo a linha mais informativa.
    if "run_id" in df.columns:
        df["_non_null_count"] = df.notna().sum(axis=1)
        df = df.sort_values("_non_null_count", ascending=False).drop_duplicates("run_id", keep="first")
        df = df.drop(columns=["_non_null_count"])

    df["model_family"] = df.apply(normalize_family, axis=1)
    df["model_signature"] = df.apply(model_signature, axis=1)

    if "num_params" in df.columns:
        df["num_params"] = pd.to_numeric(df["num_params"], errors="coerce")

    metric_col = choose_metric_column(df)
    df["metric_main"] = pd.to_numeric(df[metric_col], errors="coerce")
    df["metric_main_name"] = metric_col
    return df.sort_values(["model_family", "metric_main"], na_position="last").reset_index(drop=True)

summary = load_all_summaries(RESULTS_DIR)
metric_col = summary["metric_main_name"].dropna().iloc[0]

print("shape:", summary.shape)
print("métrica principal:", metric_col)
print("famílias:", summary["model_family"].value_counts().to_dict())
print("\ncolunas:")
print(list(summary.columns))

TABLE_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(TABLE_DIR / "summary_unificado.csv", index=False)

display(summary.head(10))


In [ ]:

# ============================================================
# SELEÇÃO DOS EXPERIMENTOS JÁ RODADOS
# ============================================================
# Este bloco NÃO treina e NÃO recalcula nada.
# Ele só filtra o summary já importado dos CSV/JSON do projeto.
#
# Edite apenas as listas abaixo para escolher quais arquiteturas
# entram nas figuras do artigo.
# ============================================================

# Mostra rapidamente o que existe nos seus arquivos importados.
def mostrar_valores_disponiveis(df: pd.DataFrame):
    cols = [c for c in [
        "model_family", "model_type", "hidden", "blocks", "n_qubits", "n_layers",
        "k", "n_vertex", "seed", "num_params", "metric_main"
    ] if c in df.columns]
    print("Colunas úteis encontradas:", cols)
    print("\nContagem por família:")
    display(df["model_family"].value_counts().rename_axis("model_family").reset_index(name="n_runs"))

    for col in ["model_type", "hidden", "blocks", "n_qubits", "n_layers", "k", "n_vertex", "seed"]:
        if col in df.columns:
            vals = pd.Series(df[col].dropna().unique())
            try:
                vals = vals.sort_values()
            except Exception:
                vals = vals.astype(str).sort_values()
            print(f"{col}: {list(vals)}")

mostrar_valores_disponiveis(summary)

# ------------------------------------------------------------
# Escolha das arquiteturas que entram nos plots.
# Tudo aqui precisa já existir em experimentos_pinn/sumario_*.csv
# ou em runs/*/*/metadata/results.json.
# ------------------------------------------------------------
SELECAO = {
    # Clássico: aqui a ideia é cortar os muito grandes.
    # Se quiser só redes pequenas, use hidden=[2,3,5] e blocks=[1,2,3,4].
    "MLP": {
        "hidden": [2, 3, 5, 10, 20],
        "blocks": [1, 2, 3, 4, 5],
    },

    # Deixe ResNet vazio/comentado se você não quiser misturar com MLP.
    # "ResNet": {
    #     "hidden": [2, 3, 5, 10],
    #     "blocks": [1, 2, 3, 4, 5],
    # },

    "QNN": {
        "n_qubits": [2, 3, 4, 7, 10],
        "n_layers": [1, 2, 3],
    },

    "CQNN": {
        "n_qubits": [2, 3, 4, 7, 10],
        "n_layers": [1, 2, 3],
    },

    "HQNN": {
        "hidden": [2, 3, 5],
        "blocks": [1, 2, 3],
        "n_qubits": [2, 3, 5],
        "n_layers": [1, 2, 3],
    },
}

# ------------------------------------------------------------
# Cortes por número de parâmetros já salvos no CSV.
# Não afeta os arquivos originais, apenas o dataframe usado nos plots.
# ------------------------------------------------------------
MAX_PARAMS_CLASSICO = 500      # coloque None para não cortar clássico; 250 para corte mais rígido
MAX_PARAMS_GERAL = None        # exemplo: 1000, ou None para não cortar geral

# Opcional: filtrar seeds específicas.
# Use None para manter todas as seeds já rodadas.
SEEDS = None
# SEEDS = [1924, 1925, 1926, 1973, 2025]


def _isin_robusto(series: pd.Series, valores) -> pd.Series:
    if valores is None:
        return pd.Series(True, index=series.index)
    if len(valores) == 0:
        return pd.Series(False, index=series.index)

    s_num = pd.to_numeric(series, errors="coerce")
    v_num = pd.to_numeric(pd.Series(valores), errors="coerce")

    # Se os valores escolhidos são numéricos, compara numericamente.
    if v_num.notna().all():
        return s_num.isin(v_num.to_list())

    return series.astype(str).isin([str(v) for v in valores])


def filtrar_por_selecao(df: pd.DataFrame, selecao: dict) -> pd.DataFrame:
    partes = []

    for model_key, regras in selecao.items():
        model_key_upper = str(model_key).upper()

        # Normalmente model_type é MLP, QNN, CQNN, HQNN.
        # Para segurança, também aceita nomes por família.
        sub = df[
            (df.get("model_type", pd.Series(index=df.index, dtype=object)).astype(str).str.upper() == model_key_upper)
            | (df.get("model_family", pd.Series(index=df.index, dtype=object)).astype(str).str.upper() == model_key_upper)
        ].copy()

        # Caso especial: MLP/ResNet pertencem à família Clássico.
        if sub.empty and model_key_upper in {"CLASSICO", "CLÁSSICO"}:
            sub = df[df["model_family"].astype(str) == "Clássico"].copy()

        for col, valores in regras.items():
            if col in sub.columns:
                sub = sub[_isin_robusto(sub[col], valores)].copy()

        partes.append(sub)

    if not partes:
        return df.iloc[0:0].copy()

    out = pd.concat(partes, ignore_index=True, sort=False)

    if "run_id" in out.columns:
        out = out.drop_duplicates("run_id", keep="first")

    return out.reset_index(drop=True)


summary_plot = filtrar_por_selecao(summary, SELECAO)

# Filtro de seeds, se ativado.
if SEEDS is not None and "seed" in summary_plot.columns:
    summary_plot = summary_plot[_isin_robusto(summary_plot["seed"], SEEDS)].copy()

# Remove clássicos grandes, se ativado.
if MAX_PARAMS_CLASSICO is not None and "num_params" in summary_plot.columns:
    mask_classico_grande = (
        summary_plot["model_family"].astype(str).eq("Clássico")
        & (pd.to_numeric(summary_plot["num_params"], errors="coerce") > MAX_PARAMS_CLASSICO)
    )
    summary_plot = summary_plot[~mask_classico_grande].copy()

# Remove qualquer modelo acima do orçamento geral, se ativado.
if MAX_PARAMS_GERAL is not None and "num_params" in summary_plot.columns:
    summary_plot = summary_plot[
        pd.to_numeric(summary_plot["num_params"], errors="coerce") <= MAX_PARAMS_GERAL
    ].copy()

summary_plot = summary_plot.reset_index(drop=True)

print("\nAntes do filtro:", summary.shape)
print("Depois do filtro:", summary_plot.shape)
print("Famílias após filtro:", summary_plot["model_family"].value_counts().to_dict())

if summary_plot.empty:
    raise ValueError(
        "O filtro deixou summary_plot vazio. Ajuste SELECAO, MAX_PARAMS_CLASSICO ou MAX_PARAMS_GERAL."
    )

# Salva a seleção usada nos plots para rastreabilidade.
summary_plot.to_csv(TABLE_DIR / "summary_selecionado_para_plots.csv", index=False)

cols_show = [c for c in [
    "model_family", "model_type", "run_id", "hidden", "blocks", "n_qubits", "n_layers",
    "k", "n_vertex", "seed", "num_params", "training_time_sec", "metric_main"
] if c in summary_plot.columns]

display(
    summary_plot[cols_show]
    .sort_values([c for c in ["model_family", "num_params", "metric_main"] if c in cols_show])
    .head(80)
)


In [ ]:

# ============================================================
# DESCOBRIR CAMINHOS DOS HISTÓRICOS DE LOSS
# ============================================================

LOSS_FILE_NAME = "loss_history_full.json"

LOSS_LABELS = {
    "Total": "Total",
    "total": "Total",
    "pde_loss": "PDE residual",
    "PDE": "PDE residual",
    "terminal_loss": "Terminal",
    "boundary_0_loss": "Boundary S=0",
    "boundary_max_loss": "Boundary S=Smax",
}

def resolve_existing_path(raw_path, project_root: Path, results_dir: Path):
    if raw_path is None or (isinstance(raw_path, float) and np.isnan(raw_path)):
        return None
    raw = str(raw_path)
    if raw.strip() == "":
        return None
    candidates = [Path(raw)]
    if not Path(raw).is_absolute():
        candidates += [project_root / raw, results_dir / raw]
    for c in candidates:
        if c.exists():
            return c
    return None

def find_loss_path(row: pd.Series) -> Path | None:
    # 1) caminho salvo pelo próprio código do projeto
    p = resolve_existing_path(row.get("loss_history_path"), PROJECT_ROOT, RESULTS_DIR)
    if p is not None:
        return p

    # 2) estrutura oficial: experimentos_pinn/runs/<model_type>/<run_id>/losses/loss_history_full.json
    run_id = str(row.get("run_id", ""))
    model_type = str(row.get("model_type", ""))
    candidates = []
    if run_id and model_type:
        candidates.append(RUNS_DIR / model_type / run_id / "losses" / LOSS_FILE_NAME)
    if run_id:
        candidates.extend(sorted(RUNS_DIR.glob(f"*/{run_id}/losses/{LOSS_FILE_NAME}")))
    for c in candidates:
        if c.exists():
            return c
    return None

def load_loss_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return {"Total": obj}
    if isinstance(obj, dict):
        return obj
    raise TypeError(f"Formato inesperado em {path}: {type(obj)}")

def sample_indices(n: int, every: int = 10, keep_log_points: bool = True):
    if n <= 0:
        return np.array([], dtype=int)
    idx = set(range(0, n, max(1, every)))
    idx.add(n - 1)
    if keep_log_points:
        logs = np.unique(np.round(np.geomspace(1, n, min(180, n))).astype(int) - 1)
        idx.update(int(i) for i in logs if 0 <= i < n)
    return np.array(sorted(idx), dtype=int)

def load_loss_long(summary_df: pd.DataFrame, every: int = 10) -> pd.DataFrame:
    rows = []
    missing = []
    for _, row in summary_df.iterrows():
        path = find_loss_path(row)
        if path is None:
            missing.append(row.get("run_id"))
            continue
        try:
            loss_obj = load_loss_json(path)
        except Exception as e:
            print(f"Falha ao ler {path}: {e}")
            continue

        for loss_name, values in loss_obj.items():
            if not isinstance(values, list) or len(values) == 0:
                continue
            arr = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
            idx = sample_indices(len(arr), every=every)
            for i in idx:
                rows.append({
                    "run_id": row.get("run_id"),
                    "model_type": row.get("model_type"),
                    "model_family": row.get("model_family"),
                    "model_signature": row.get("model_signature"),
                    "seed": row.get("seed", np.nan),
                    "num_params": row.get("num_params", np.nan),
                    "metric_main": row.get("metric_main", np.nan),
                    "epoch": int(i + 1),
                    "loss_name_raw": loss_name,
                    "loss_name": LOSS_LABELS.get(loss_name, loss_name),
                    "loss_value": arr[i],
                    "loss_path": str(path),
                })
    out = pd.DataFrame(rows)
    if missing:
        print(f"Histórico de loss não encontrado para {len(missing)} runs. Exemplos:", missing[:5])
    if out.empty:
        print("Nenhum histórico de loss foi carregado.")
        return out
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=["loss_value", "epoch"])
    out.to_csv(TABLE_DIR / "loss_long_sampled.csv", index=False)
    return out

# Ajuste LOSS_EVERY se quiser curvas mais/menos densas.
LOSS_EVERY = 10
loss_long_plot = load_loss_long(summary_plot, every=LOSS_EVERY)
loss_long = loss_long_plot  # alias para compatibilidade

print("loss_long_plot shape:", loss_long_plot.shape)
if not loss_long_plot.empty:
    print(loss_long_plot["loss_name"].value_counts().to_dict())
    display(loss_long_plot.head())



## 1. Curva de convergência: loss total por família

Esta é a primeira figura que conversa diretamente com os dois artigos: antes de mostrar preço ou Greeks, mostramos se cada família realmente converge na PDE de Black--Scholes e como a convergência muda entre clássico, QNN, CQNN e híbrido.


In [ ]:

def plot_total_loss_by_family(loss_df: pd.DataFrame, y_quantiles=(0.25, 0.75), path_name="fig_01_total_loss_by_family"):
    if loss_df.empty:
        print("Sem loss_long para plotar.")
        return None
    d = loss_df[loss_df["loss_name"] == "Total"].copy()
    d = d[d["loss_value"] > 0]
    if d.empty:
        print("Não encontrei loss Total positiva.")
        return None

    g = (
        d.groupby(["model_family", "epoch"], as_index=False)
         .agg(
             median_loss=("loss_value", "median"),
             q_low=("loss_value", lambda x: np.nanquantile(x, y_quantiles[0])),
             q_high=("loss_value", lambda x: np.nanquantile(x, y_quantiles[1])),
             n=("loss_value", "size"),
         )
    )

    fig, ax = plt.subplots(figsize=(6.4, 4.1))
    families = [f for f in ["Clássico", "QNN", "CQNN", "Híbrido"] if f in g["model_family"].unique()]
    for fam in families:
        s = g[g["model_family"] == fam].sort_values("epoch")
        ax.plot(s["epoch"], s["median_loss"], label=fam)
        ax.fill_between(s["epoch"].to_numpy(), s["q_low"].to_numpy(), s["q_high"].to_numpy(), alpha=0.16, linewidth=0)

    ax.set_yscale("log")
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.legend(loc="best")
    finish_axes(ax, xlabel="Epoch", ylabel="Total loss", title="Convergência da loss total por família", logy=True)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, ax

plot_total_loss_by_family(loss_long_plot)
plt.show()



## 2. Decomposição da loss nas melhores runs

Aqui a ideia é checar se o modelo está reduzindo apenas a condição terminal/fronteira ou se também está respeitando o residual da PDE. Para PINN financeiro, isso é importante porque a loss total mistura termos de natureza diferente.


In [ ]:

def get_best_runs(summary_df: pd.DataFrame, top_per_family: int = 1) -> pd.DataFrame:
    d = summary_df.dropna(subset=["metric_main"]).copy()
    if d.empty:
        return pd.DataFrame()
    return (
        d.sort_values("metric_main")
         .groupby("model_family", as_index=False, group_keys=False)
         .head(top_per_family)
         .sort_values(["model_family", "metric_main"])
    )

best_runs = get_best_runs(summary_plot, top_per_family=1)
best_runs.to_csv(TABLE_DIR / "best_run_per_family.csv", index=False)
display(best_runs[[c for c in ["model_family", "run_id", "model_type", "model_signature", "num_params", "metric_main", "training_time_sec"] if c in best_runs.columns]])

def plot_components_for_best_runs(loss_df: pd.DataFrame, best_df: pd.DataFrame, path_name="fig_02_loss_components_best_runs"):
    if loss_df.empty or best_df.empty:
        print("Sem dados suficientes.")
        return None
    run_ids = set(best_df["run_id"].astype(str))
    d = loss_df[loss_df["run_id"].astype(str).isin(run_ids)].copy()
    d = d[d["loss_value"] > 0]
    if d.empty:
        print("Sem dados de loss para as melhores runs.")
        return None

    families = list(best_df["model_family"].unique())
    n = len(families)
    fig, axes = plt.subplots(1, n, figsize=(4.4 * n, 3.7), squeeze=False, sharey=True)

    component_order = ["Total", "PDE residual", "Terminal", "Boundary S=0", "Boundary S=Smax"]
    for ax, fam in zip(axes[0], families):
        rid = str(best_df[best_df["model_family"] == fam]["run_id"].iloc[0])
        sub = d[d["run_id"].astype(str) == rid]
        for comp in component_order:
            s = sub[sub["loss_name"] == comp].sort_values("epoch")
            if s.empty:
                continue
            ax.plot(s["epoch"], s["loss_value"], label=comp)
        ax.set_yscale("log")
        finish_axes(ax, xlabel="Epoch", ylabel="Loss", title=fam, logy=True)
        ax.legend(loc="best")

    fig.suptitle("Decomposição da loss nas melhores runs", y=1.03)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, axes

plot_components_for_best_runs(loss_long_plot, best_runs)
plt.show()



## 3. Métricas de teste por família

Esta parte é o equivalente direto à comparação de desempenho entre famílias: clássico versus circuitos quânticos e híbridos. A métrica usada é a coluna encontrada no projeto, normalmente `mse_teste_desnormalizado`.


In [ ]:

def plot_error_distribution(summary_df: pd.DataFrame, path_name="fig_03_error_distribution"):
    d = summary_df.dropna(subset=["metric_main"]).copy()
    if d.empty:
        print("Sem metric_main.")
        return None

    families = [f for f in ["Clássico", "QNN", "CQNN", "Híbrido"] if f in d["model_family"].unique()]
    data = [d[d["model_family"] == f]["metric_main"].dropna().to_numpy() for f in families]

    fig, ax = plt.subplots(figsize=(5.8, 4.0))
    bp = ax.boxplot(data, labels=families, patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], WOLFRAM_COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.30)
        patch.set_edgecolor("black")
    for key in ["whiskers", "caps", "medians"]:
        for artist in bp[key]:
            artist.set_color("black")
            artist.set_linewidth(0.9)

    # pontos individuais com jitter
    rng = np.random.default_rng(123)
    for i, vals in enumerate(data, start=1):
        x = i + rng.normal(0, 0.045, size=len(vals))
        ax.scatter(x, vals, s=10, alpha=0.45, edgecolor="none")

    ax.set_yscale("log")
    finish_axes(ax, xlabel="Família", ylabel=metric_col, title="Distribuição do erro de teste", logy=True)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, ax

plot_error_distribution(summary_plot)
plt.show()



## 4. Erro versus número de parâmetros

Essa é uma das figuras mais importantes para ligar o seu projeto ao artigo de QPINNs: não basta mostrar que o erro é baixo; precisamos mostrar a relação **erro × capacidade/quantidade de parâmetros**.


In [ ]:

def plot_error_vs_params(summary_df: pd.DataFrame, path_name="fig_04_error_vs_params"):
    if "num_params" not in summary_df.columns:
        print("Coluna num_params não existe no resumo.")
        return None
    d = summary_df.dropna(subset=["metric_main", "num_params"]).copy()
    d = d[(d["metric_main"] > 0) & (d["num_params"] > 0)]
    if d.empty:
        print("Sem dados para erro vs parâmetros.")
        return None

    fig, ax = plt.subplots(figsize=(6.1, 4.1))
    families = [f for f in ["Clássico", "QNN", "CQNN", "Híbrido"] if f in d["model_family"].unique()]
    for fam in families:
        s = d[d["model_family"] == fam]
        ax.scatter(
            s["num_params"], s["metric_main"],
            s=28, alpha=0.72, edgecolor="black", linewidth=0.25, label=fam,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.legend(loc="best")
    finish_axes(ax, xlabel="Número de parâmetros treináveis", ylabel=metric_col, title="Erro versus parâmetros", logy=True)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, ax

plot_error_vs_params(summary_plot)
plt.show()



## 5. Fronteira de eficiência: melhores modelos por orçamento de parâmetros

Aqui filtramos os modelos que ficam na fronteira: menor erro para dado número de parâmetros. Essa figura ajuda a argumentar sobre expressividade/custo, que é central na comparação PINN clássico × QPINN/HPINN.


In [ ]:

def pareto_front(df: pd.DataFrame, x="num_params", y="metric_main") -> pd.DataFrame:
    d = df.dropna(subset=[x, y]).copy()
    d = d[(d[x] > 0) & (d[y] > 0)].sort_values([x, y])
    best = np.inf
    keep = []
    for _, row in d.iterrows():
        if row[y] < best:
            keep.append(True)
            best = row[y]
        else:
            keep.append(False)
    return d.loc[keep]

def plot_pareto(summary_df: pd.DataFrame, path_name="fig_05_pareto_efficiency"):
    if "num_params" not in summary_df.columns:
        print("Coluna num_params não existe.")
        return None
    d = summary_df.dropna(subset=["metric_main", "num_params"]).copy()
    d = d[(d["metric_main"] > 0) & (d["num_params"] > 0)]
    if d.empty:
        return None

    fig, ax = plt.subplots(figsize=(6.1, 4.1))
    families = [f for f in ["Clássico", "QNN", "CQNN", "Híbrido"] if f in d["model_family"].unique()]
    for fam in families:
        s = d[d["model_family"] == fam]
        ax.scatter(s["num_params"], s["metric_main"], s=22, alpha=0.35, edgecolor="none", label=fam)

    pf = pareto_front(d)
    ax.plot(pf["num_params"], pf["metric_main"], color="black", marker="o", linewidth=1.4, label="Fronteira")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.legend(loc="best")
    finish_axes(ax, xlabel="Número de parâmetros treináveis", ylabel=metric_col, title="Fronteira erro-parâmetros", logy=True)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, ax

plot_pareto(summary_plot)
plt.show()

display(pareto_front(summary_plot)[[c for c in ["model_family", "run_id", "model_signature", "num_params", "metric_main"] if c in summary_plot.columns]].head(20))



## 6. Tempo de treino versus erro

Essa figura é importante porque os artigos de QPINNs normalmente discutem expressividade, mas o custo computacional também precisa aparecer. Aqui fica claro se o ganho de erro vem com custo de tempo muito maior.


In [ ]:

def plot_time_vs_error(summary_df: pd.DataFrame, path_name="fig_06_time_vs_error"):
    if "training_time_sec" not in summary_df.columns:
        print("Coluna training_time_sec não existe.")
        return None
    d = summary_df.dropna(subset=["metric_main", "training_time_sec"]).copy()
    d = d[(d["metric_main"] > 0) & (d["training_time_sec"] > 0)]
    if d.empty:
        print("Sem dados para tempo vs erro.")
        return None

    fig, ax = plt.subplots(figsize=(6.1, 4.1))
    families = [f for f in ["Clássico", "QNN", "CQNN", "Híbrido"] if f in d["model_family"].unique()]
    for fam in families:
        s = d[d["model_family"] == fam]
        ax.scatter(s["training_time_sec"], s["metric_main"], s=28, alpha=0.72, edgecolor="black", linewidth=0.25, label=fam)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.legend(loc="best")
    finish_axes(ax, xlabel="Tempo de treino [s]", ylabel=metric_col, title="Tempo de treino versus erro", logy=True)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, ax

plot_time_vs_error(summary_plot)
plt.show()



## 7. Mapas de hiperparâmetros

Estes heatmaps dão a visão mais próxima das figuras de grade dos artigos: para QNN/CQNN/Híbrido, ajudam a discutir qubits, camadas, correlatores e estabilidade; para o clássico, mostram o papel de largura e profundidade.


In [ ]:

def plot_heatmap_table(table: pd.DataFrame, x: str, y: str, value: str, title: str, path_name: str):
    pivot = table.pivot_table(index=y, columns=x, values=value, aggfunc="median")
    pivot = pivot.sort_index(ascending=True).sort_index(axis=1, ascending=True)
    if pivot.empty:
        print("Pivot vazio:", title)
        return None

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    im = ax.imshow(pivot.values, origin="lower", aspect="auto", cmap=WOLFRAM_TEMPERATURE)

    ax.set_xticks(np.arange(pivot.shape[1]))
    ax.set_xticklabels([str(c) for c in pivot.columns])
    ax.set_yticks(np.arange(pivot.shape[0]))
    ax.set_yticklabels([str(i) for i in pivot.index])

    # anota valores em notação científica curta
    vals = pivot.values.astype(float)
    finite = vals[np.isfinite(vals)]
    if finite.size:
        threshold = np.nanmedian(finite)
    else:
        threshold = np.nan
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = vals[i, j]
            if np.isfinite(v):
                txt = f"{v:.1e}" if abs(v) < 0.01 or abs(v) >= 100 else f"{v:.3g}"
                ax.text(j, i, txt, ha="center", va="center", fontsize=8, color="black")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.035)
    cbar.outline.set_linewidth(0.7)
    finish_axes(ax, xlabel=x, ylabel=y, title=title)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, ax

def plot_project_heatmaps(summary_df: pd.DataFrame):
    outputs = []

    # Clássico: hidden x blocks
    d = summary_df[summary_df["model_family"] == "Clássico"].copy()
    if not d.empty and {"hidden", "blocks"}.issubset(d.columns):
        outputs.append(plot_heatmap_table(d, x="hidden", y="blocks", value="metric_main", title="Clássico: erro mediano por hidden × blocks", path_name="fig_07a_heatmap_classico"))

    # QNN: n_qubits x n_layers
    d = summary_df[summary_df["model_family"] == "QNN"].copy()
    if not d.empty and {"n_qubits", "n_layers"}.issubset(d.columns):
        outputs.append(plot_heatmap_table(d, x="n_qubits", y="n_layers", value="metric_main", title="QNN: erro mediano por qubits × layers", path_name="fig_07b_heatmap_qnn"))

    # CQNN: um heatmap por k, se k existir
    d = summary_df[summary_df["model_family"] == "CQNN"].copy()
    if not d.empty and {"n_vertex", "n_layers"}.issubset(d.columns):
        if "k" in d.columns and d["k"].notna().any():
            for k_val in sorted(pd.to_numeric(d["k"], errors="coerce").dropna().unique()):
                dk = d[pd.to_numeric(d["k"], errors="coerce") == k_val]
                outputs.append(plot_heatmap_table(dk, x="n_vertex", y="n_layers", value="metric_main", title=f"CQNN: erro mediano por vertices × layers, k={int(k_val)}", path_name=f"fig_07c_heatmap_cqnn_k{int(k_val)}"))
        else:
            outputs.append(plot_heatmap_table(d, x="n_vertex", y="n_layers", value="metric_main", title="CQNN: erro mediano por vertices × layers", path_name="fig_07c_heatmap_cqnn"))

    # Híbrido: n_qubits x n_layers
    d = summary_df[summary_df["model_family"] == "Híbrido"].copy()
    if not d.empty and {"n_qubits", "n_layers"}.issubset(d.columns):
        outputs.append(plot_heatmap_table(d, x="n_qubits", y="n_layers", value="metric_main", title="Híbrido: erro mediano por qubits × layers", path_name="fig_07d_heatmap_hibrido"))

    return outputs

plot_project_heatmaps(summary_plot)
plt.show()



## 8. Estabilidade por seed

Como seus scripts rodam várias seeds, esta figura resume a variabilidade por arquitetura. Isso é bom para o artigo porque evita escolher uma única run isolada.


In [ ]:

def summarize_by_signature(summary_df: pd.DataFrame) -> pd.DataFrame:
    d = summary_df.dropna(subset=["metric_main"]).copy()
    if d.empty:
        return pd.DataFrame()
    agg = (
        d.groupby(["model_family", "model_signature"], as_index=False)
         .agg(
             median_error=("metric_main", "median"),
             mean_error=("metric_main", "mean"),
             std_error=("metric_main", "std"),
             min_error=("metric_main", "min"),
             max_error=("metric_main", "max"),
             n_runs=("metric_main", "size"),
             median_params=("num_params", "median") if "num_params" in d.columns else ("metric_main", "size"),
             median_time=("training_time_sec", "median") if "training_time_sec" in d.columns else ("metric_main", "size"),
         )
         .sort_values("median_error")
    )
    return agg

signature_summary = summarize_by_signature(summary_plot)
signature_summary.to_csv(TABLE_DIR / "signature_summary.csv", index=False)
display(signature_summary.head(20))

def plot_top_signatures(sig_df: pd.DataFrame, top_n=15, path_name="fig_08_top_signatures_stability"):
    if sig_df.empty:
        print("Sem signature_summary.")
        return None
    d = sig_df.sort_values("median_error").head(top_n).copy()
    d = d.sort_values("median_error", ascending=True)
    labels = d["model_signature"].astype(str).to_list()
    y = np.arange(len(d))

    fig, ax = plt.subplots(figsize=(7.2, max(4.2, 0.33 * len(d))))
    ax.errorbar(
        d["median_error"], y,
        xerr=d["std_error"].fillna(0),
        fmt="o", color="black", ecolor="0.45", elinewidth=0.9, capsize=2,
    )

    # colore ponto por família
    fam_to_color = {fam: WOLFRAM_COLORS[i % len(WOLFRAM_COLORS)] for i, fam in enumerate(["Clássico", "QNN", "CQNN", "Híbrido"])}
    for i, row in d.reset_index(drop=True).iterrows():
        ax.scatter(row["median_error"], i, s=42, color=fam_to_color.get(row["model_family"], "0.4"), edgecolor="black", linewidth=0.25, zorder=3)

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xscale("log")
    ax.invert_yaxis()
    finish_axes(ax, xlabel=f"Erro mediano por seed ({metric_col})", ylabel="Arquitetura", title=f"Top {top_n} arquiteturas por estabilidade", logy=False)
    fig.tight_layout()
    savefig(fig, path_name)
    return fig, ax

plot_top_signatures(signature_summary, top_n=15)
plt.show()



## 9. Tabelas finais desta etapa

Essas tabelas ficam salvas em `tables_loss_mathematica_style/` e podem entrar direto como apoio para escrever a seção experimental.


In [ ]:

# Top runs individuais
cols = [
    "model_family", "run_id", "model_type", "model_signature", "seed",
    "num_params", "training_time_sec", "metric_main",
    "final_loss_total", "final_loss_pde_loss", "final_loss_terminal_loss",
    "final_loss_boundary_0_loss", "final_loss_boundary_max_loss",
]
cols = [c for c in cols if c in summary_plot.columns]

top_runs = summary_plot.dropna(subset=["metric_main"]).sort_values("metric_main").head(30)
top_runs[cols].to_csv(TABLE_DIR / "top_30_runs.csv", index=False)
display(top_runs[cols])

# Melhor por família
best_by_family = get_best_runs(summary_plot, top_per_family=5)
best_by_family[cols].to_csv(TABLE_DIR / "best_5_by_family.csv", index=False)
display(best_by_family[cols])

print("\nFiguras em:", FIG_DIR)
print("Tabelas em:", TABLE_DIR)
